In [1]:
import batting_statlines as stat 
import pandas as pd 
import numpy as np 
import mysql.connector as sql
from IPython import display
from collections import namedtuple

pd.options.display.max_columns = None

In [2]:
fangraphs_board = pd.read_csv('pitching_1947_2001.csv')
fangraphs_board.rename(columns={'season': 'yearID', 'team': 'teamID'}, inplace = True)
fangraphs_board.drop(['G', 'GS', 'HR/FB', 'xFIP', 'xFIP-', 'playerid', 'K/9', 'BB/9', 'K/9+', 'BB/9+'], axis=1, inplace=True)
fangraphs_board = stat.order_by(fangraphs_board, ['yearID', 'teamID'], True)
fangraphs_board = fangraphs_board[fangraphs_board.teamID != '- - -']
for col in ['K%', 'BB%']:
    fangraphs_board[col] = fangraphs_board[col].replace('%', '', regex=True)
    fangraphs_board[col] = pd.to_numeric(fangraphs_board[col])

## Here We Go!

The first thing that I want to do, is to group everybody into teams. In order to do this analysis, I'm going to be looking at Johnson and Schilling's numbers together as a single aggregate. I think that would be the best, becuase I'm going to compare him to other top 2 pitchers on teams so we're really comparing "groups" of players rather than individual players.

### The Plan

What I plan on doing is first, create the aggregate values for each column, for the top two pitchers in a team each year. After I do that I want to rank each pair of staff aces for each statistic. From there, create an average score for how they rank in each statistic to find the most "dominant" pairing of pitchers. Then, look across each year and see who has the highest "dominance" score.

In [3]:
years = fangraphs_board.yearID.unique()
teams = fangraphs_board.teamID.unique()
dont_do = ['name', 'teamID']
team_dict = namedtuple('team_avgs', 'col_name dict')
def make_avgs_dict(year, team):
    team_dict = {}
    team_df = fangraphs_board[(fangraphs_board.yearID == year) & (fangraphs_board.teamID == team)]
    team_df = stat.order_by(team_df, 'WAR', False)
    team_df = team_df.head(2)
    for col in team_df.columns:
        if col not in dont_do:
            team_dict[col] = team_df[col].mean()
    return team_dict

dicts = {}
for year in years:
    for team in teams:
        years_teams = fangraphs_board[fangraphs_board.yearID == year].teamID.unique()
        if team in years_teams:
            t = team_dict(col_name=f'{team}_{year}', dict=make_avgs_dict(year, team))
            dicts[t.col_name] = t.dict


In [4]:
top2_aggs = pd.DataFrame(data=dicts.values(), index=dicts.keys())
top2_aggs.reset_index(inplace=True)
top2_aggs['index'] = top2_aggs['index'].apply(lambda x: x.split('_')[0])
top2_aggs.rename(columns={'index': 'teamID'}, inplace=True)
top2_aggs

,teamID,yearID,W,L,IP,WHIP,K%,BB%,HR/9,BABIP,ERA,FIP,ERA-,FIP-,K/BB+,WHIP+,K%+,BB%+,WAR
0,Athletics,1947.0,15.5,10.0,251.65,1.315,9.45,10.50,0.485,0.2500,3.015,3.780,81.5,100.0,91.5,94.0,97.0,105.5,2.90
1,Braves,1947.0,21.0,11.0,277.60,1.215,11.10,7.10,0.555,0.2580,2.925,3.285,72.5,83.0,155.5,84.5,116.5,75.0,5.50
2,Browns,1947.0,8.0,14.5,185.10,1.470,11.30,8.15,0.610,0.3035,4.695,3.480,120.5,87.5,143.0,105.0,116.0,81.5,3.20
3,Cardinals,1947.0,15.0,6.5,196.05,1.350,12.50,7.40,0.430,0.2990,3.105,3.030,74.5,74.5,168.5,94.0,131.5,78.5,4.45
4,Cubs,1947.0,10.5,15.0,195.00,1.390,10.25,8.50,0.640,0.2785,3.800,3.665,97.0,95.5,119.0,96.5,107.5,90.0,2.55
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1781,Mariners,2020.0,2.0,1.0,17.20,0.740,22.40,3.00,2.040,0.1520,3.060,4.550,78.0,113.0,307.0,57.0,96.0,31.0,0.20
1782,Rockies,2020.0,2.5,0.5,18.00,1.035,25.05,6.80,0.500,0.2540,2.270,2.495,47.5,56.5,137.5,83.5,104.5,76.0,0.65
1783,Diamondbacks,2020.0,1.0,0.5,17.60,1.110,25.50,5.90,1.760,0.2680,2.550,4.260,61.0,102.5,335.5,89.5,106.5,66.0,0.20
1784,Nationals,2020.0,0.5,0.5,12.60,1.120,35.95,7.95,1.080,0.3095,3.145,2.585,73.5,61.0,349.5,90.5,150.0,89.0,0.40


In [14]:
year_2001 = top2_aggs[top2_aggs.yearID == 2001]
year_2001.drop('BABIP', axis=1, inplace=True)
year_2001.head()

,teamID,yearID,W,L,IP,WHIP,K%,BB%,HR/9,ERA,FIP,ERA-,FIP-,K/BB+,WHIP+,K%+,BB%+,WAR
1209,Athletics,2001.0,19.5,8.5,232.05,1.190,17.50,6.35,0.700,3.410,3.475,78.0,79.0,139.5,85.5,106.0,76.5,5.40
1210,Braves,2001.0,14.5,11.5,226.05,1.115,19.70,5.35,0.735,3.045,3.255,71.0,74.5,216.0,81.5,109.5,62.0,5.70
1211,Cardinals,2001.0,19.0,9.5,221.60,1.275,19.55,6.35,0.705,3.125,3.395,73.0,77.0,147.0,93.0,108.5,74.0,5.30
1212,Cubs,2001.0,16.0,6.0,203.10,1.205,22.40,8.35,0.900,3.580,3.650,83.5,83.0,142.0,88.0,124.5,97.5,4.15
1213,Dodgers,2001.0,15.0,10.0,234.00,1.150,22.30,9.00,0.880,3.350,3.850,82.0,88.0,118.0,84.0,124.0,105.0,4.30


In [18]:
counting_cols = ['W', 'IP', 'K%', 'K/BB+', 'WAR', 'K%+']
def rank_it(df, counting):
    ranked_df = df.copy()
    for col in ranked_df.columns:
        if col == 'teamID' or col == 'yearID':
            continue
        elif col in counting:
            ranked_df[col] = df[col].rank(method='first', ascending=False)
        else:
            ranked_df[col] = df[col].rank(method='first')
    return ranked_df
rank_2001 = rank_it(year_2001, counting_cols)
rank_2001['composite'] = rank_2001.filter(regex = '[^yearID]').mean(axis=1)
rank_2001 = stat.order_by(rank_2001, 'composite', True)
rank_2001

,teamID,yearID,W,L,IP,WHIP,K%,BB%,HR/9,ERA,FIP,ERA-,FIP-,K/BB+,WHIP+,K%+,BB%+,WAR,composite
0,Diamondbacks,2001.0,1.0,3.0,1.0,1.0,1.0,6.0,17.0,1.0,1.0,1.0,1.0,1.0,2.0,1.0,5.0,1.0,2.7500
1,Yankees,2001.0,7.0,13.0,11.0,9.0,7.0,1.0,3.0,10.0,2.0,8.0,2.0,2.0,8.0,4.0,1.0,2.0,5.6250
2,Braves,2001.0,13.0,20.0,6.0,4.0,8.0,4.0,4.0,2.0,3.0,2.0,3.0,3.0,4.0,8.0,4.0,3.0,5.6875
3,Athletics,2001.0,2.0,6.0,3.0,7.0,12.0,9.0,1.0,7.0,5.0,7.0,5.0,8.0,7.0,11.0,10.0,4.0,6.5000
4,Cardinals,2001.0,3.0,8.0,9.0,12.0,9.0,10.0,2.0,3.0,4.0,3.0,4.0,6.0,12.0,9.0,8.0,5.0,6.6875
5,Mariners,2001.0,4.0,2.0,7.0,3.0,17.0,7.0,8.0,4.0,9.0,5.0,10.0,12.0,3.0,15.0,9.0,9.0,7.7500
6,White Sox,2001.0,6.0,5.0,10.0,2.0,23.0,5.0,15.0,5.0,16.0,4.0,11.0,11.0,1.0,21.0,6.0,10.0,9.4375
7,Cubs,2001.0,5.0,1.0,15.0,10.0,3.0,21.0,12.0,11.0,7.0,12.0,7.0,7.0,10.0,5.0,21.0,8.0,9.6875
8,Dodgers,2001.0,10.0,10.0,2.0,5.0,4.0,24.0,10.0,6.0,10.0,10.0,9.0,15.0,6.0,6.0,24.0,7.0,9.8750
9,Twins,2001.0,8.0,22.0,5.0,6.0,25.0,2.0,14.0,9.0,13.0,6.0,12.0,4.0,5.0,23.0,2.0,11.0,10.4375
